## 1. 配置区 ⚠️ 必须修改

> 把下方 `OBS_BUCKET` 改成你自己的桶名；换区域时同步改 `OBS_ENDPOINT`。
> Notebook 绑定了 OBS 委托的话，AK/SK 留空即可（moxing 自动使用委托）；
> 没有委托就临时填自己的 AK/SK——**用完即清空，绝不要把真实密钥提交进仓库或分享出去**。


In [ ]:
# ==================== ⚠️ 必须修改 ====================
OBS_BUCKET   = "<你的桶名>"                             # 你的 OBS 桶名
OBS_PREFIX   = "models"                                # OBS 存储路径前缀（= 服务端 OBS_KEY 的前半段）
OBS_ENDPOINT = "obs.cn-north-4.myhuaweicloud.com"      # OBS endpoint（换区域同步改，部署侧 OBS_ENDPOINT 也要改）

# IAM 访问密钥（华为云控制台 → 我的凭证 → 访问密钥）
# ⚠️ Notebook 绑定了 OBS 委托就留空（moxing 自动用委托）；没有委托才临时填自己的
# ⚠️ 真实 AK/SK 用完即清空，绝不要提交进仓库或分享出去
ACCESS_KEY_ID     = ""    # ← 填你的 AK（或留空用委托）
SECRET_ACCESS_KEY = ""    # ← 填你的 SK（或留空用委托）
# ==========================================================

import os
from pathlib import Path

# 工作目录：ModelArts Notebook 用自带可写目录，本地运行自动落到当前目录
WORK_DIR = "/home/ma-user/work/xgb_train" if Path("/home/ma-user").exists() else "."

assert "<" not in OBS_BUCKET, "请先把 <你的桶名> 换成实际桶名"

WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
OLD_DIR = WORK / "model_out" / "old"
NEW_DIR = WORK / "model_out" / "new"
OLD_DIR.mkdir(parents=True, exist_ok=True)
NEW_DIR.mkdir(parents=True, exist_ok=True)

# 本地保留两套模型（用于对比验证）
OLD_MODEL_LOCAL = OLD_DIR / "xgboost_breast_cancer.json"
NEW_MODEL_LOCAL = NEW_DIR / "xgboost_breast_cancer.json"

# OBS 只有一个目标路径（不分 old/new 子目录，靠 §7 的 ACTIVE_MODEL 切换）
ACTIVE_MODEL_OBS = f"obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json"

print(f"工作目录:      {WORK}")
print(f"OBS 桶:        {OBS_BUCKET}")
print(f"OBS 目标路径:  {ACTIVE_MODEL_OBS}")


## 2. 安装依赖 + 导入

ModelArts Notebook 自带 `xgboost`、`scikit-learn`、`pandas`。
我们额外确认 `moxing`（华为云 OBS 操作库）可用。


In [ ]:
# === 修复 ModelArts Notebook 的 pandas ABI 冲突 ===
# 根因：~/modelarts-dev/modelarts-sdk/ 里捆绑了一个 pandas 副本，被插到 sys.path
# 最前面，它的 C 扩展与环境的 numpy 版本不匹配，导致
#   ValueError: numpy.dtype size changed (Expected 96, got 88)
# 修复：把该目录从 sys.path 剔除，并清理已缓存的 pandas 模块，强制回退到
# conda site-packages 里和 numpy 匹配的正常 pandas。
import sys as _sys

_BAD_FRAGMENTS = ("modelarts-dev/modelarts-sdk", "modelarts-dev\\modelarts-sdk")
_original_path = list(_sys.path)
_sys.path = [p for p in _sys.path if not any(frag in p.replace("\\", "/") for frag in _BAD_FRAGMENTS)]
if len(_sys.path) != len(_original_path):
    removed = set(_original_path) - set(_sys.path)
    print(f"[path-fix] 已从 sys.path 剔除: {removed}")

# 清理可能已被错误 pandas 污染的缓存模块
for _mod_name in list(_sys.modules):
    if _mod_name == "pandas" or _mod_name.startswith("pandas."):
        _mod = _sys.modules[_mod_name]
        _mod_file = getattr(_mod, "__file__", "") or ""
        if any(frag in _mod_file.replace("\\", "/") for frag in _BAD_FRAGMENTS):
            del _sys.modules[_mod_name]
            print(f"[path-fix] 已卸载缓存模块: {_mod_name}")
# === path-fix 结束 ===

# === 安装缺失依赖（ModelArts 镜像可能没预装 scikit-learn / xgboost）===
import subprocess as _sp
def _pip_install(*pkgs):
    print(f"[install] pip install {' '.join(pkgs)} ...")
    _sp.check_call([_sys.executable, "-m", "pip", "install", "--quiet", *pkgs])

for _pkg, _import_name in [("scikit-learn", "sklearn"), ("xgboost", "xgboost")]:
    try:
        __import__(_import_name)
        print(f"[install] {_pkg} 已安装，跳过")
    except ImportError:
        _pip_install(_pkg)
# === 安装结束 ===

import json
import shutil
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

print(f"pandas 来源: {pd.__file__}")
print(f"pandas 版本: {pd.__version__}")
print(f"numpy  版本: {np.__version__}")

# ModelArts 自带的 OBS 操作库
try:
    import moxing as mox
    print(f"moxing 版本: {mox.__version__ if hasattr(mox, '__version__') else 'unknown'}")
    # 如果提供了 AK/SK，设置认证（否则使用 Notebook 委托）
    if ACCESS_KEY_ID and SECRET_ACCESS_KEY:
        import moxing.framework.content_db as content_db
        content_db.configure_obs_credentials(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            endpoint=OBS_ENDPOINT,
        )
        print("已使用 AK/SK 配置 moxing 认证")
    else:
        print("未提供 AK/SK，将使用 Notebook 委托认证")
    HAS_MOXING = True
except ImportError:
    HAS_MOXING = False
    print("⚠️ moxing 不可用，将尝试 esdk-obs-python 作为 fallback")

print(f"\nxgboost: {__import__('xgboost').__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")


## 3. 连接 MRS Hive（Kerberos 安全集群）

训练数据不再用 sklearn 自带数据集，而是从 MRS Hive 的 `breast_cancer` 表读取（569 行 x 31 列，建表见 `hive_export/breast_cancer_hive.sql`）。

以下 6 个格子复制自 `hive_export/modelarts_hive_conn.ipynb` 第 1–6 格（实测配方，排障速查与原理见该 notebook 和 `hive_export/MRS_RUN.md`、`docs/adr/0002`）：

> ⚠️ 重启内核后，第 1、3、4 格必须重跑（PATH / KRB5_CONFIG 都在内存里）；票据 24h 过期后重跑第 5 格（重新输密码）。

In [ ]:
# ================== 1. 连接配置（实测值，换集群时改这里） ==================
HIVE_HOST = "10.0.0.15"    # HiveServer2 内网 IP（master1）
HIVE_PORT = 21066          # HiveServer2 Thrift 端口
DATABASE  = "default"
USERNAME  = "hhx"          # MRS 业务用户

REALM    = "252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM"  # MRS 系统域名(Realm)
SPN_HOST = "hadoop." + REALM.lower()   # 实测正确的 SPN 中间段（haddop_ 变体是错的）

KDC_HOSTS = ["10.0.0.15", "10.0.0.51"]  # 两个 Master 都写，容错
KDC_PORT  = 21732                        # 华为 MRS 专用 KDC 端口，不是 88！

print("principal =", f"hive/{SPN_HOST}@{REALM}")

In [ ]:
# ================== 2. 网络探测（安全组没放行在这里快速暴露） ==================
import socket

def probe(host, port, name, timeout=5):
    s = socket.socket(); s.settimeout(timeout)
    try:
        s.connect((host, port)); print(f"[OK]   {name} {host}:{port} 可达")
        return True
    except Exception as e:
        print(f"[FAIL] {name} {host}:{port} 不可达: {e}")
        return False
    finally:
        s.close()

net_ok = probe(HIVE_HOST, HIVE_PORT, "HiveServer2")
for k in KDC_HOSTS:
    net_ok &= probe(k, KDC_PORT, "KDC")

assert net_ok, (
    "网络不通：请确认 notebook 与 MRS 同 VPC，且安全组放行 21066 与 21732(TCP+UDP)。\n"
    "只通 21066 不够 —— kinit 还要访问 KDC 的 21732。"
)

In [ ]:
# ================== 3. 环境预装（幂等；自动适配三种环境；实时进度） ==================
# 目标产物: kinit 二进制 + cyrus 的 sasl(GSSAPI 插件在位) + 纯 python 的 pyhive 等
#   ★ 集群 qop=auth-conf，必须用 cyrus 的 sasl；pure-sasl+pykerberos 实测在
#     加密包装阶段报 "Invalid token was supplied"（见 ADR-0002）
# 适配顺序（打印 [env] 说明命中哪支）：
#   A. root            -> apt 装 gcc/g++/krb5-user/头文件 + GSSAPI 插件, pip 编译 sasl
#   B. ma-user+免密sudo -> 同 A，apt 前加 sudo -n
#   C. 无 root(常见)   -> conda-forge 预编译: krb5(自带 kinit) + sasl，无需编译器
import collections, importlib, os, re, shutil, subprocess, sys, threading, time

def have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

# ---- 实时进度执行器: 关键行即时打印(带耗时), 静默期打心跳, 失败回放末尾输出 ----
_BAR = re.compile(r"^\W*\[\W*\d+%\W*\]\W*$")            # apt 的 [ 12%] 进度条(噪声)
_HOT = re.compile(r"solving|collecting|downloading|extracting|preparing|executing|"
                  r"transaction|unpacking|setting up|processing|fetched|^get|^hit|"
                  r"building wheel|successfully|installed|nothing to do|all requested|"
                  r"error|fail|conflict|warn", re.I)     # 值得展示的进度/结果行

def run_stream(cmd, note=None, heartbeat=20):
    """流式执行外部命令。返回码 0=成功; 失败时自动回放末尾 40 行输出。"""
    if note: print(f"[run] {note}", flush=True)
    t0, tail = time.time(), collections.deque(maxlen=40)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    stop = threading.Event()
    def _beat():                                          # 长时间静默时证明"还活着"
        quiet = time.time()
        while not stop.wait(2):
            if time.time() - quiet >= heartbeat:
                print(f"   ... {int(time.time()-t0)}s 仍在运行（{cmd[0]} 无新输出，静默属正常）", flush=True)
                quiet = time.time()
    th = threading.Thread(target=_beat, daemon=True); th.start()
    for line in proc.stdout:
        line = line.rstrip()
        tail.append(line)
        if line and not _BAR.match(line) and _HOT.search(line):
            print(f"[{int(time.time()-t0):>3}s] {line}", flush=True)
    rc = proc.wait(); stop.set(); th.join(timeout=1)
    if rc != 0:
        print("---- 命令末尾输出（最多 40 行）----")
        print("\n".join(t for t in tail if t.strip()) or "(无输出)")
    return rc

# conda 装的 kinit 在 sys.prefix/bin：内核重启后 PATH 可能不含它，先补上，
# 否则依赖已齐也会误判 need_kinit，白白再跑一次 conda 求解（本实例实测约 5 分钟）
if os.path.isfile(os.path.join(sys.prefix, "bin", "kinit")):
    os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
need_kinit, need_sasl = shutil.which("kinit") is None, not have("sasl")

# --- 3.1 系统层 ---
if need_kinit or need_sasl:
    apt, env_name = None, "无 root（走 conda 分支）"
    if os.geteuid() == 0:
        apt, env_name = ["apt-get"], "root"
    else:
        sudo_ok = subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0
        if sudo_ok:
            apt, env_name = ["sudo", "-n", "apt-get"], "ma-user + 免密 sudo"
    print(f"[env] {env_name}", flush=True)

    if apt is not None:
        # libsasl2-modules-gssapi-mit / libsasl2-modules = cyrus 的 GSSAPI 插件(必须!)
        if run_stream(apt + ["update"], "apt-get update") != 0:
            raise SystemExit("[FAIL] apt-get update 失败")
        if run_stream(apt + ["install", "-y", "gcc", "g++", "krb5-user",
                             "libkrb5-dev", "libsasl2-dev",
                             "libsasl2-modules-gssapi-mit", "libsasl2-modules"],
                      "apt 安装编译链 + krb5 + cyrus sasl（首次约 1-2 分钟）") != 0:
            raise SystemExit("[FAIL] apt install 失败")
    else:
        # 无 root：ModelArts 自带 anaconda，conda-forge 有预编译的 krb5 和 sasl
        conda = shutil.which("conda")
        assert conda, "[FAIL] 没找到 conda —— 请把报错反馈给维护者"
        print("[env] 无 root —— 走 conda-forge 预编译路径", flush=True)
        # --prefix sys.prefix: 显式装进当前内核环境，避免落到 base
        # --override-channels: 只用本命令指定的频道 —— 实例的 condarc 若配了已失效的
        #   镜像频道(如 TUNA 的 anaconda/pkgs/free, 已停同步 404), 不加这个会一起失败
        base = [conda, "install", "-y", "--override-channels", "--prefix", sys.prefix]
        attempts = [
            (base + ["-c", "conda-forge", "krb5", "sasl"],
             "conda 安装 krb5 + sasl（conda-forge，绕开失效镜像频道；求解+下载 1-3 分钟）"),
            (base + ["-c", "https://conda.anaconda.org/conda-forge", "krb5", "sasl"],
             "conda 重试（官方 conda-forge 源直连，可能较慢）"),
        ]
        rc = 1
        for cmd, note in attempts:
            rc = run_stream(cmd, note)
            if rc == 0:
                break
        if rc != 0:
            raise SystemExit("[FAIL] conda install 两个源均失败（实例镜像源不可用？）；"
                             "备选：改用 OBS 离线 wheel，或把上面的报错反馈给维护者")
        # conda 装的 kinit 在 $CONDA_PREFIX/bin，放进 PATH 供第 5 格使用
        os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
        print("[提示] 若下方 3.3 自检 import 报错（conda 刚装完包内核未感知），"
              "重启内核后重跑第 1、3、4 格即可（已装的会自动跳过）")
else:
    print("[env] 系统依赖已齐（kinit + sasl），跳过安装")

# --- 3.2 python 包（纯 python，pip 即可） ---
PIP_PKGS = [p for p, m in (("pyhive", "pyhive"), ("thrift", "thrift"),
                           ("thrift-sasl", "thrift_sasl"), ("sasl", "sasl"))
            if not have(m)]
if PIP_PKGS:
    if run_stream([sys.executable, "-m", "pip", "install"] + PIP_PKGS,
                  f"pip 安装 {PIP_PKGS}") != 0:
        raise SystemExit("[FAIL] pip install 失败")

# --- 3.3 自检：import + cyrus 的 GSSAPI 插件在位（auth-conf 的硬前提） ---
# 注意: cyrus 的 sasl 包没有"列出机制"的 API（available_mechs 是 pure-sasl 的
# 接口，误用会 AttributeError）。改用功能探测: 真的 init + start 一次 GSSAPI，
# 与第 6 格连接时是同一条代码路径（pyhive.get_sasl_client -> setAttr+init；
# thrift_sasl.open -> start）:
#   start 成功                                  -> 插件在位（且已有票据）
#   报 No worthy mechs / No mechanism available -> 插件缺失（依赖没装全，fatal）
#   报 GSSAPI 凭据类错误（无票据等）           -> 插件在位，第 5 格 kinit 后即可用
import glob
from pyhive import hive
from pyhive.hive import get_installed_sasl
import thrift_sasl
import sasl as cyrus_sasl

_p = cyrus_sasl.Client()
_p.setAttr("host", SPN_HOST)          # 第 1 格的 SPN 中间段，仅作 SASL 层参数
_p.setAttr("service", "hive")
assert _p.init(), f"cyrus sasl 初始化失败: {_p.getError()!r}"
_ok, _mech, _resp = _p.start("GSSAPI")
_err = _p.getError()
_err = _err.decode("utf-8", "replace") if isinstance(_err, bytes) else (_err or "")
if _ok:
    print("[OK] python 依赖就绪；GSSAPI 插件可用；kinit =", shutil.which("kinit"))
elif "worthy mechs" in _err.lower() or "no mechanism available" in _err.lower():
    for _pat in (os.path.join(sys.prefix, "lib*", "sasl2", "*"),
                 "/usr/lib/*/sasl2/*", "/usr/lib64/sasl2/*"):
        for _h in glob.glob(_pat):
            if "gssapi" in os.path.basename(_h).lower():
                print("  gssapi 插件文件:", _h)
    raise SystemExit(
        f"cyrus sasl 缺 GSSAPI 插件（{_err}）\n"
        "root/sudo 环境: 检查 libsasl2-modules-gssapi-mit 是否装上；\n"
        "conda 环境: 把 !ls $CONDA_PREFIX/lib/sasl2/ 的输出发给维护者排查")
else:
    print("[OK] python 依赖就绪；GSSAPI 插件在位（暂无票据，第 5 格 kinit 后生效；"
          f"探测信息: {_err.splitlines()[0] if _err else '-'}）")
    print("kinit =", shutil.which("kinit"))


In [ ]:
# ================== 4. 生成 krb5.conf 并生效 ==================
# dns_canonicalize_hostname=false 是关键：SPN 中间段 hadoop.xxx 是 DNS 里
# 不存在的"假域名"，必须禁止 Kerberos 客户端解析它，原样当 SPN 用。
# udp_preference_limit=1 让 AS/TGS 请求走 TCP —— 与第 2 格探测的 TCP 端口一致。
import os
from pathlib import Path

KRB5_FILE = Path.cwd() / "krb5.conf"
lines = [
    "[libdefaults]",
    f"    default_realm = {REALM}",
    "    dns_canonicalize_hostname = false",   # <- 关键
    "    rdns = false",
    "    udp_preference_limit = 1",
    "",
    "[realms]",
    f"    {REALM} = {{",
    *[f"        kdc = {h}:{KDC_PORT}" for h in KDC_HOSTS],
    f"        admin_server = {KDC_HOSTS[0]}:{KDC_PORT}",
    "    }",
    "",
    "[domain_realm]",
    f"    .{REALM.lower()} = {REALM}",
    f"    {SPN_HOST} = {REALM}",
    f"    .{SPN_HOST} = {REALM}",
    "",
]
KRB5_FILE.write_text("\n".join(lines), encoding="utf-8")
os.environ["KRB5_CONFIG"] = str(KRB5_FILE)   # 后面 kinit 与 cyrus-sasl 都读它
print(f"[OK] 已生成 {KRB5_FILE} 并设置 KRB5_CONFIG\n")
print("\n".join(lines))

In [ ]:
# ================== 5. kinit 获取用户票据（TGT，24h 有效） ==================
# 已有票据则跳过；否则弹出密码输入框（getpass，不在代码里留明文）。
# kinit 可能来自 apt(krb5-user, /usr/bin) 或 conda(krb5, $CONDA_PREFIX/bin)，自适应。
import getpass, shutil, subprocess

KINIT = shutil.which("kinit") or os.path.join(sys.prefix, "bin", "kinit")
KLIST = shutil.which("klist") or os.path.join(sys.prefix, "bin", "klist")

def _run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True,
                          env={**os.environ, "KRB5_CONFIG": str(KRB5_FILE)}, **kw)

r = _run([KLIST])
if r.returncode == 0 and "krbtgt" in r.stdout:
    print("[OK] 已有有效票据，跳过 kinit：")
    print("\n".join(r.stdout.splitlines()[:4]))
else:
    principal = f"{USERNAME}@{REALM}"
    pw = getpass.getpass(f"输入 {principal} 的密码: ")
    r = _run([KINIT, principal], input=pw + "\n")
    assert r.returncode == 0, f"[FAIL] kinit 失败（密码错/KDC 不通？）：{r.stderr.strip()}"
    print(f"[OK] kinit 成功: {principal}")

In [ ]:
# ================== 6. 连接 HiveServer2（核心：解耦 TCP 地址与 SPN） ==================
# pyhive 在 auth=KERBEROS 时把 TCP 连接的 host 直接当 SPN 的 host 用 -> 必然对不上
# （会去 KDC 请求 hive/10.0.0.15@REALM 的票据，而集群注册的是固定串 SPN）。
# 官方逃生口 thrift_transport=...：TCP 层连内网 IP，SASL 层 host 传 SPN 中间段。
# get_installed_sasl 在装了 cyrus 的 sasl 包后自动优先用它（qop=auth-conf 必需）。
from thrift.transport import TSocket

def make_transport():
    tcp = TSocket.TSocket(HIVE_HOST, HIVE_PORT)
    tcp.setTimeout(30000)
    sasl_factory = lambda: get_installed_sasl(
        host=SPN_HOST, sasl_auth="GSSAPI", service="hive")
    return thrift_sasl.TSaslClientTransport(sasl_factory, "GSSAPI", tcp)

PRINCIPAL = f"hive/{SPN_HOST}@{REALM}"
print("尝试 SPN:", PRINCIPAL)
conn = hive.connect(thrift_transport=make_transport(),
                    database=DATABASE, username=USERNAME)
print("[OK] 连接成功！生效 SPN =", PRINCIPAL)

## 4. 从 Hive 加载数据 + 样本定义

从 MRS Hive 的 `breast_cancer` 表读取数据集（替代 sklearn 的 `load_breast_cancer`），并把 Hive 的下划线列名还原成 sklearn 带空格的特征名，随后嵌入 `sample_request.json` 的测试样本（用于验证新旧模型给出不同预测值）。

> 取数用小块 fetch（`cur.arraysize = 5`）：整表一次取回会触发部分环境 libsasl2 的大帧解密 bug（见 cell 内注释）。


In [ ]:
# === 从 Hive 读取乳腺癌数据集（替代 sklearn 的 load_breast_cancer）===
# 表结构见 hive_export/breast_cancer_hive.sql：569 行 x 31 列（30 特征 + target）
cur = conn.cursor()
# 坑（实测）：pyhive 默认 arraysze=10000，fetchall 会把 569 行打进一个超大 SASL
# 加密帧；部分环境的 libsasl2（2.1.28 有已知回归 bug）解不开大帧，报
#   TTransportException: sasl_decode ... Unable to find a callback: 32775
# 对策：小块取，每次 5 行 —— 与 conn notebook 里 LIMIT 5 稳定通过是同一原理。
cur.arraysize = 5
cur.execute("SELECT * FROM breast_cancer")
cols = [d[0].split(".")[-1] for d in cur.description]   # 去掉可能带的 库名.表名. 前缀
try:
    rows = cur.fetchall()
except Exception as e:
    if "sasl_decode" in str(e) or "32775" in str(e):
        msg = """小块取仍触发 sasl 解包失败（libsasl2 2.1.28 已知回归 bug）。
修复：新格子里跑下面两行，装完点菜单 Kernel - Restart Kernel，从头重跑：
  import subprocess
  subprocess.run(['conda', 'install', '-y', '-c', 'conda-forge',
                  '--override-channels', 'libsasl2=2.1.27'], check=True)"""
        raise SystemExit(msg) from e
    raise
finally:
    cur.close()
print(f"取回 {len(rows)} 行")

df_hive = pd.DataFrame(rows, columns=cols)
# Hive 列名是下划线风格（mean_radius），改回 sklearn 的带空格风格（"mean radius"），
# 与 sample_request.json / app.py 的特征名保持一致
df_hive.columns = [c.replace("_", " ") for c in df_hive.columns]

X = df_hive.drop(columns=["target"])
y = df_hive["target"].astype(int)
print(f"Hive breast_cancer: {X.shape[0]} 样本, {X.shape[1]} 特征")

# === 数据校验 ===
# 特征名只借自 sklearn（不用它的数据），保证与推理服务的特征顺序严格一致
from sklearn.datasets import load_breast_cancer
FEATURE_NAMES = list(load_breast_cancer().feature_names)
assert list(X.columns) == FEATURE_NAMES, f"列名与 sklearn 特征名不一致: {list(X.columns)[:3]} ..."
assert X.shape == (569, 30), f"期望 569x30, 实际 {X.shape}"
assert set(y.unique()) <= {0, 1}, f"target 取值异常: {sorted(y.unique())}"
print("[OK] 校验通过：569x30、特征名与 sklearn 一致、target ∈ {0,1}")

# === 嵌入测试样本（来自 sample_request.json）===
sample_row = {
    "mean radius": 17.99, "mean texture": 10.38, "mean perimeter": 122.8,
    "mean area": 1001.0, "mean smoothness": 0.1184, "mean compactness": 0.2776,
    "mean concavity": 0.3001, "mean concave points": 0.1471,
    "mean symmetry": 0.2419, "mean fractal dimension": 0.07871,
    "radius error": 1.095, "texture error": 0.9053, "perimeter error": 8.589,
    "area error": 153.4, "smoothness error": 0.006399,
    "compactness error": 0.04904, "concavity error": 0.05373,
    "concave points error": 0.01587, "symmetry error": 0.03003,
    "fractal dimension error": 0.006193, "worst radius": 25.38,
    "worst texture": 17.33, "worst perimeter": 184.6, "worst area": 2019.0,
    "worst smoothness": 0.1622, "worst compactness": 0.6656,
    "worst concavity": 0.7119, "worst concave points": 0.2654,
    "worst symmetry": 0.4601, "worst fractal dimension": 0.1189,
}
sample_df = pd.DataFrame([sample_row], columns=FEATURE_NAMES)
print(f"测试样本: {len(sample_row)} 特征")


## 5. 训练函数

训练 → 评估 → 保存到本地 → 打印样本预测值。
新旧模型对同一样本预测值不同，这正是后面热切换验证的判定依据。


In [ ]:
def train_and_save(params, random_state, output_path, label):
    """训练模型，评估并保存，返回样本预测概率。"""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y,
    )
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=random_state,
        **params,
    )
    model.fit(X_train, y_train, verbose=False)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    pred = float(model.predict_proba(sample_df)[0, 1])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(output_path))
    size = output_path.stat().st_size

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  超参:     {params}")
    print(f"  Accuracy: {acc:.4f}  AUC: {auc:.4f}")
    print(f"  样本预测: {pred:.16f}")
    print(f"  本地保存: {output_path} ({size:,} bytes)")
    return pred


## 6. 训练 OLD 模型（baseline）

100 棵树，浅深度，高学习率。


In [ ]:
old_pred = train_and_save(
    params=dict(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
    ),
    random_state=42,
    output_path=OLD_MODEL_LOCAL,
    label="OLD MODEL (baseline)",
)


## 7. 训练 NEW 模型（不同超参）

250 棵树，深深度，低学习率，加正则化。


In [ ]:
new_pred = train_and_save(
    params=dict(
        n_estimators=250, max_depth=6, learning_rate=0.01,
        subsample=0.6, colsample_bytree=0.5,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0, gamma=0.5,
    ),
    random_state=2024,
    output_path=NEW_MODEL_LOCAL,
    label="NEW MODEL (updated)",
)


## 8. 上传模型到 OBS 🚀

把训练好的模型上传到 OBS 的**单一目标路径**（推理服务只认这个路径）：

- `obs://{OBS_BUCKET}/{OBS_PREFIX}/xgboost_breast_cancer.json`

> 推理服务（`app.py`，环境变量 `OBS_BUCKET` + `OBS_KEY`）从该路径读取模型。
> 想切换 old / new：改下方 `ACTIVE_MODEL` 后重跑本 cell 即可。


In [ ]:
# 选择当前要上传哪套模型到 OBS（改成 "new" 可切换为新模型）
ACTIVE_MODEL = "old"   # "old" 或 "new"

LOCAL_TO_UPLOAD = OLD_MODEL_LOCAL if ACTIVE_MODEL == "old" else NEW_MODEL_LOCAL
print(f"当前选择: {ACTIVE_MODEL} 模型")
print(f"本地文件: {LOCAL_TO_UPLOAD} ({LOCAL_TO_UPLOAD.stat().st_size:,} bytes)")
print(f"OBS 目标: {ACTIVE_MODEL_OBS}")
print()

def upload_to_obs(local_path, obs_uri, label=""):
    """上传单个文件到 OBS（覆盖写）。"""
    tag = f" [{label}]" if label else ""
    print(f"  上传{tag}: {local_path} → {obs_uri}")

    if HAS_MOXING:
        mox.file.copy(str(local_path), obs_uri)
    else:
        # fallback: esdk-obs-python
        from obs import ObsClient
        assert ACCESS_KEY_ID and SECRET_ACCESS_KEY, "AK/SK 必填（moxing 不可用时）"
        client = ObsClient(
            access_key_id=ACCESS_KEY_ID,
            secret_access_key=SECRET_ACCESS_KEY,
            server=f"https://{OBS_ENDPOINT}",
        )
        key = obs_uri.replace(f"obs://{OBS_BUCKET}/", "")
        resp = client.putFile(OBS_BUCKET, key, str(local_path))
        assert resp.status < 300, f"上传失败: status={resp.status}"
        client.close()

    size = local_path.stat().st_size
    print(f"    ✅ 完成 ({size:,} bytes)")

upload_to_obs(LOCAL_TO_UPLOAD, ACTIVE_MODEL_OBS, ACTIVE_MODEL.upper())
print(f"\n🚀 上传完成！")
print(f"   推理服务 app.py 会从 {ACTIVE_MODEL_OBS} 读取模型。")
print(f"   切换模型：把 ACTIVE_MODEL 改成 'new'，重跑这个 cell 即可。")
